In [2]:
from PyPDF2 import PdfReader
import os
import shutil
from PIL import Image
import os
import fitz
from docx import Document
from docx.shared import Inches
import io
from PIL import Image
import os

In [3]:
def extract_pic(file):
    doc = fitz.open(file)
    file_name = file.replace(".pdf", "")
    
    image_xrefs = {}

    for page in doc:
        for image in page.get_images():
            image_xrefs.setdefault(image[0])  
    
    images = []
    sizes = []
    sources = []
    
    for index, xref in enumerate(image_xrefs):
        img = doc.extract_image(xref)
        image_bytes = img['image']
        
        size = f"{img['width']} x {img['height']}"
        
        images.append(Image.open(io.BytesIO(image_bytes)))
        sizes.append(size)
        sources.append(file_name)
        
    return images, sizes, sources

In [4]:
def make_rows_bold(*rows):
    for row in rows:
        for cell in row.cells:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.bold = True

In [9]:
doc1 = Document()
example = extract_pic("articles/Akcija Extinction Rebellion Zagreb_ Blokirali ulaz u LNG terminal na Krku _ NACIONAL.HR _ News portal najutjecajnijeg političkog tjednika.pdf")

table = doc1.add_table(rows=1, cols=4, style = 'TableGrid')

row = table.rows[0].cells
row[0].text = 'id'
row[1].text = 'Image'
row[2].text = 'Length/Width'
row[3].text = 'Source'

num_row = 0 

images = example[0]
sizes = example[1]
sources = example[2]
    
for a in range(len(images)):
    image_filename = f"test_image_{a}.png"
    example[0][a].save(image_filename)
    
    file_size = os.path.getsize(image_filename) // 1000
    
    if file_size >= 10:
    
        row_cells = table.add_row().cells
        paragraph = row_cells[0].paragraphs[0]
        paragraph.add_run(text=str(num_row))

        paragraph = row_cells[1].paragraphs[0]
        run = paragraph.add_run()
        run.add_picture(image_filename, width=Inches(1.5))

        paragraph = row_cells[2].paragraphs[0]
        paragraph.add_run(text=sizes[a])

        paragraph = row_cells[3].paragraphs[0]
        paragraph.add_run(text=sources[a])

        num_row += 1
        
    else:
        continue
    if os.path.exists(f"test_image_{a}.png"):
            os.remove(f"test_image_{a}.png")
    
make_rows_bold(table.rows[0])

doc1.save("output/Example_Images.docx")

In [7]:
print(len(images))

5


In [ ]:
f = [
    os.path.join(root, file)
    for root, dirs, files in os.walk('./articles')
    for file in files
]

pdf_list = []
for ff in f:
    if ff.endswith('.pdf'):
        pdf_list.append(ff)
print(len(pdf_list))

37


In [11]:
word_document = Document()
table = word_document.add_table(rows=1, cols=4, style = 'TableGrid')
row = table.rows[0].cells 
row[0].text = 'id'
row[1].text = 'Image'
row[2].text = 'Length/Width'
row[3].text = 'Source'

num_row = 0 

for i in range(len(pdf_list)):
    images = extract_pic(pdf_list[i])[0]
    sizes = extract_pic(pdf_list[i])[1]
    sources = extract_pic(pdf_list[i])[2]
    
    for a in range(len(images)):
        image_filename = f"image_{a}.png"
        images[a].save(image_filename)
        
        file_size = os.path.getsize(image_filename) // 1000
    
        if file_size >= 10:
        
            row_cells = table.add_row().cells
            paragraph = row_cells[0].paragraphs[0]
            paragraph.add_run(text=str(num_row))

            paragraph = row_cells[1].paragraphs[0]
            run = paragraph.add_run()
            run.add_picture(image_filename, width=Inches(1.5))

            paragraph = row_cells[2].paragraphs[0]
            paragraph.add_run(text=sizes[a])

            paragraph = row_cells[3].paragraphs[0]
            paragraph.add_run(text=sources[a])

            num_row += 1
        else:
            continue
        if os.path.exists(f"image_{a}.png"):
            os.remove(f"image_{a}.png")

    make_rows_bold(table.rows[0])
        
word_document.save('output/table.docx') 